In [10]:
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import  PyPDFLoader
from langchain_community.vectorstores import  FAISS
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from langchain_core.prompts import  PromptTemplate
from langchain_core.output_parsers import  JsonOutputParser
from langchain_classic.chains.summarize import load_summarize_chain
from pydantic import  BaseModel, Field
from helper_function import *


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [39]:
pdf_path = r"c:\\Users\\Anupam\\Desktop\\AI AGENTS\\SOPHISTICATED-AGENT\\Harry_Potter_Book_1_The_Sorcerers_Stone.pdf"

Splitting the Pdf into Chapters

In [3]:
print(f"File Exists at {os.path.abspath('./Harry_Potter_Book_1_The_Sorcerers_Stone.pdf')}")
# Check 2: File size?
print(f"2. File size: {os.path.getsize('./Harry_Potter_Book_1_The_Sorcerers_Stone.pdf') if os.path.exists('./Harry_Potter_Book_1_The_Sorcerers_Stone.pdf') else 'N/A'} bytes")

File Exists at c:\Users\Anupam\Desktop\AI AGENTS\SOPHISTICATED-AGENT\Harry_Potter_Book_1_The_Sorcerers_Stone.pdf
2. File size: 3188138 bytes


In [4]:
chapters = split_into_chapters(r"c:\\Users\\Anupam\\Desktop\\AI AGENTS\\SOPHISTICATED-AGENT\\Harry_Potter_Book_1_The_Sorcerers_Stone.pdf")
print(f"Total Chapters: {len(chapters)}")
chapters = replace_t_with_space(chapters)
print(f"Total Chapters after replacing newlines: {len(chapters)}")

Total pages: 740
Extracted text length: 1094550 characters
First 500 chars:   Harry Potter
and the Goblet Of Fire
 
 
by
J. K. Rowling
Illustrations by Mary Grandpré
 
 
 
 
Arthur A. Levine Books
An Imprint of Scholastic Press To Peter Rowling,
In Memory of Mr. Ridley
And to Susan Sladden,
Who helped Harry
Out of his cupboard Text copyright © 2000 by J.K. Rowling
Illustrations by Mary GrandPre copyright © 2000 Warner
Bros.
All rights reserved. Published by Scholastic Press, a division
of Scholastic Inc.,
Publishers since 1920.
SCHOLASTIC, SCHOLASTIC PRESS, and the LANT
Total Chapters: 37
Total Chapters after replacing newlines: 37


Create aa List of Quotes from the books

In [6]:
loader = PyPDFLoader(r"c:\\Users\Anupam\\Desktop\\AI AGENTS\\SOPHISTICATED-AGENT\\Harry_Potter_Book_1_The_Sorcerers_Stone.pdf")
documents = loader.load()
cleaned_docs = replace_t_with_space(documents)
book_quotes_list = extract_book_quotes_as_documents(cleaned_docs)
print(f"Total Quotes extracted: {len(book_quotes_list)}")

Total Quotes extracted: 3214


Create Chapter Summaries using LLMs

In [7]:
summarization_prompt_template = """
Write an extensive summary of the following {text}.
Summary:""
"""
summarization_prompt = PromptTemplate(template=summarization_prompt_template,input_variables=["text"])

In [32]:
import transformers
def create_chapter_summary(chapter):
    """
    Creates a summary of a chapter using a large language model (LLM).

    Args:
        chapter: A Document object representing the chapter to summarize.

    Returns:
        A Document object containing the summary of the chapter.
    """
    llm = ChatOllama(model="llama3.2:latest")
    chain = load_summarize_chain(llm, chain_type="stuff",prompt=summarization_prompt)
    summary = chain.invoke([chapter])
    return Document(page_content=summary["output_text"], metadata={"chapter": chapter.metadata["chapter"] })

In [33]:
chapters_summary =[]
for chapter in chapters:
    summary_doc = create_chapter_summary(chapter)
    chapters_summary.append(summary_doc)

In [34]:
chapters_summary[0]

Document(metadata={'chapter': 1}, page_content="The story begins with Frank Bryce, an ordinary man who has just discovered that two wizards are secretly planning to kill his son, Harry Potter. Frank sneaks into their hideout to listen and discovers a conversation between the two wizards, Lord Voldemort and Wormtail, who reveal that they have been using a Memory Charm on someone named Bertha Jorkins to extract information from her.\n\nAs Frank listens in, he realizes that Voldemort is planning to kill his son Harry. He tries to leave but gets frozen in fear by a giant snake that appears out of nowhere and follows the sound of Voldemort's voice. The snake passes through the gap where Voldemort was speaking, leaving Frank shaken.\n\nSuddenly, Voldemort emerges from behind the armchair and reveals himself to Frank, who is terrified. However, just as Voldemort raises his wand to cast a killing curse, the story is interrupted by Harry Potter waking up with a start in his own bed two hundred 

Encode a Book into a Vector Store uisng Ollama

In [ ]:
def encode_book(pdf_path, chunk_size = 1000, chunk_overlap = 200):
    """
    Encodes a PDF book into a vector store using OpenAI embeddings.

    Args:
        path: The path to the PDF file.
        chunk_size: The desired size of each text chunk.
        chunk_overlap: The amount of overlap between consecutive chunks.

    Returns:
        A FAISS vector store containing the encoded book content.
    """
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
     # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(documents)
    cleaned_docs = replace_t_with_space(chunks)
    embeddings = OllamaEmbeddings(model="llama3.2:latest")
    vector_store = FAISS.from_documents(cleaned_docs, embeddings)
    return vector_store

Encoding Chapter Summaries into Vector Store

In [ ]:
def encode_chapter_summaries(chapter_summary):
    """
    Encodes a list of chapter summaries into a vector store using OpenAI embeddings.

    Args:
        chapter_summaries: A list of Document objects representing the chapter summaries.

    Returns:
        A FAISS vector store containing the encoded chapter summaries.
    """

    embeddings = OllamaEmbeddings(model="llama3.2:latest")  # Create Ollama embeddings
    chapter_summaries_vectorstore = FAISS.from_documents(chapter_summary, embeddings)  # Create vector store
    return chapter_summaries_vectorstore

Encoding Quotes into Vector Store

In [37]:
def encode_quotes(book_quotes_list):
    """
    Encodes a list of quotes into a vector store using OpenAI embeddings.

    Args:
        quote_list: A list of quotes.

    Returns:
        A FAISS vector store containing the encoded quotes.
    """

    embeddings = OllamaEmbeddings(model="llama3.2:latest")  # Create Ollama embeddings
    quotes_vectorstore = FAISS.from_documents(book_quotes_list, embeddings)  # Create vector store
    return quotes_vectorstore

Creating vector stores and retrievers

In [40]:
# ### IF VECTOR STORES ALREADY EXIST, LOAD THEM
if os.path.exists("chunks_vector_store") and os.path.exists("chapter_summaries_vector_store") and os.path.exists("book_quotes_vectorstore"):
    embeddings = OllamaEmbeddings(model="llama3.2:latest")
    chunks_vector_store =  FAISS.load_local("chunks_vector_store", embeddings, allow_dangerous_deserialization=True)
    chapter_summaries_vector_store =  FAISS.load_local("chapter_summaries_vector_store", embeddings, allow_dangerous_deserialization=True)
    book_quotes_vectorstore =  FAISS.load_local("book_quotes_vectorstore", embeddings, allow_dangerous_deserialization=True)

else:
    # Encode the book and chapter summaries
    chunks_vector_store = encode_book(pdf_path, chunk_size=1000, chunk_overlap=200)
    chapter_summaries_vector_store = encode_chapter_summaries(chapters_summary)
    book_quotes_vectorstore = encode_quotes(book_quotes_list)


    # Save the vector stores
    chunks_vector_store.save_local("chunks_vector_store")
    chapter_summaries_vector_store.save_local("chapter_summaries_vector_store")
    book_quotes_vectorstore.save_local("book_quotes_vectorstore")

Create a retriever from the vector store

In [ ]:
chunks_query_retriever = chunks_vector_store.as_retriever(search_kwargs={"k": 1})     
chapter_summaries_query_retriever = chapter_summaries_vector_store.as_retriever(search_kwargs={"k": 1})
book_quotes_query_retriever = book_quotes_vectorstore.as_retriever(search_kwargs={"k": 10})